# **Trabalho Final POO - Análise de Sentimentos**
## **Disciplina: Programação Orientada a Objetos**
## **Professor: Dirson**
## **Turma: D01**

## **Alunos:**
**- Gustavo Rodrigues Ribeiro / RA:202003570** \
**- Breno Machado Barros / RA:202014607** \

# **Domínio do Negócio: E-commerce**
### Para este projeto, o domínio de negócio escolhido será o de e-commerce com foco na análise de sentimentos em avaliações de produtos. As avaliações são obtidas de um dataset público de avaliações de produtos eletrônicos, como o Amazon Product Reviews Dataset, que oferece avaliações reais em várias categorias de produtos.

## Iniciando o PySpark

Esta célula de código instala o Spark no ambiente de execução Colab. Aqui está uma explicação passo a passo:

1. **`!apt-get install openjdk-11-jdk-headless -qq > /dev/null`**: este comando instala o OpenJDK 11 (versão headless, sem interface gráfica), que é um requisito para o Spark. O `-qq` suprime a saída e o `> /dev/null` redireciona a saída para o nada, tornando o processo mais silencioso.

2. **`!wget -q https://dlcdn.apache.org/spark/spark-3.5.2/spark-3.5.3-bin-hadoop3.tgz`**: Este comando baixa o arquivo compactado do Spark 3.5.2 (construído para o Hadoop 3) do site oficial do Apache Spark. O `-q` suprime a saída de download.

3. **`!tar xf spark-3.5.3-bin-hadoop3.tgz`**: Este comando extrai o arquivo compactado baixado do Spark, criando um diretório chamado `spark-3.5.3-bin-hadoop3`.

4. **`!pip -q install findspark`**: Este comando instala a biblioteca `findspark` usando `pip`. Findspark é uma biblioteca Python que torna mais fácil configurar o Spark em um ambiente Python, principalmente no Colab. Ela define as variáveis de ambiente necessárias para que o Spark funcione corretamente.

Após executar essas linhas, você terá o Spark instalado e pronto para ser usado em seu notebook Colab.

In [ ]:
!apt-get install openjdk-11-jdk-headless -qq > /dev/null
!wget -q https://dlcdn.apache.org/spark/spark-3.5.3/spark-3.5.3-bin-hadoop3.tgz
!tar xf spark-3.5.3-bin-hadoop3.tgz
!pip -q install findspark

Defina as variáveis de ambiente do Spark:

In [ ]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.3-bin-hadoop3"

O código a seguir garante que o Spark seja configurado corretamente e esteja pronto para uso em seu ambiente Python.

* **`findspark.init()`**: executa a função `init()` do módulo `findspark`. Esta função:
    * Localiza a instalação do Spark em seu sistema.
    * Configura as variáveis de ambiente necessárias para que o Python possa interagir com o Spark. Isso permite que o driver Python (seu código Python) se comunique com o executor Spark (o código que realmente processa os dados).


In [ ]:
import findspark
findspark.init()

Depois de executar a célula anterior, você poderá importar e usar as bibliotecas Spark como `pyspark.sql.SparkSession` para criar uma sessão Spark e começar a trabalhar com dados.

**OBS: Vale lembra que esse é um código para a criação de um modelo de treinamento e teste de IA para análise de sentimento através de um dataset de avaliações de produtos, com cerca de 6000000 de reviews. Logo, é importante entender que apenas o Colab (versão gratuita) não possui recursos computacionais (GPU e RAM) suficientes para executar o modelo por completo. Assim, recomendamos a utilização da máquina local com cerca de 64gb de RAM ou uma máquina virtual como a N-highmem-64gb no Dataproc do Google Cloud Console, com o ambiente virtual Jupyter Notebook. Assim, o código irá executar sem erros de memória ou GPU.**

**OBS 2: Caso utilize uma máquina virtual como a n2-highmem-8 no Dataproc do Google Cloud Console, com o ambiente virtual Jupyter Notebook, não serão necessários os passos acima, apenas continue daqui.**

**OBS 3: No caso da Camada Silver, o código atual, ele pode ser executado no Google Colab sem problemas, assim irá funcionar corretamente.**

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = SparkSession.builder.appName('Trabalho Final Silver').master("local[*]").getOrCreate()

# Caso necessário utilize spark.stop() para encerrar a sessão atual e reiniciar a SparkSession com as configurações abaixo (Caso utilize uma máquina virtual como a n2-highmem-8 no Dataproc do Google Cloud Console)
# spark.stop()
# spark = SparkSession.builder.appName('Trabalho Final Silver').config("spark.driver.memory", "64g").config("spark.executor.memory", "64g").config("spark.executor.cores", "8").master("local[*]").getOrCreate()

print("Versão do Spark:", spark.version)

Versão do Spark: 3.5.3


## **Arquitetura Medallion: SILVER**
## **Processamento e Limpeza dos Dados**

Aqui estaremos sincronizando nossa conta no Drive ao ambiente Colab, para que os arquivos em nuvem sejam gerenciados (lidos e escritos) e manipulados diretamente no Drive.

**OBS: Caso esteja utilizando o Dataproc do Google Cloud Console, com o ambiente virtual Jupyter Notebook, você podera utilizar o Data Lake Google Cloud Storage (GCS) que está conectado a sua conta, não necessitando desse processo de sincronização com o Drive.**

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
!ls /content/drive

MyDrive  Shareddrives


In [ ]:
# ---------------------------
# Camada Silver: Dados Limpos e Enriquecidos
# ---------------------------

# Carregar dados da camada Bronze no Drive ou GCS
dataset_path = "/content/drive/MyDrive/Disciplinas-UFG/PDM/TrabalhoFinal/ArquiteturaMedallion/Bronze/reviews_bronze"  # Mude para o diretório desejado

# Ler os dados diretamente da camada Bronze no Drive ou GCS em .parquet
reviews_df_silver = spark.read.parquet(dataset_path)

#### -------------------------------------- Limpeza de Dados ---------------------------------------- ####

# 1. Filtra para remover as linhas onde style não é NULL
reviews_df_silver = reviews_df_silver.filter(col("style")["Format:"].isNull())

# 2. Tratar valores ausentes (Exemplo: Preencher 'vote' com 0 se estiver ausente)
reviews_df_silver = reviews_df_silver.fillna({'vote': 0, 'reviewText': "Missing Field", 'summary': "Missing Field"})

# 3. Remover espaços extras e vírgula para transformar '01 1, 2008' em '01-01-2008' ou '11 21, 2013' em '11-21-2013'
reviews_df_silver = reviews_df_silver.withColumn("reviewTime", regexp_replace("reviewTime", r"(\d{2}) (\d), (\d{4})", "$1-0$2-$3"))
reviews_df_silver = reviews_df_silver.withColumn("reviewTime", regexp_replace("reviewTime", r"(\d{2}) (\d{2}), (\d{4})", "$1-$2-$3"))

# 4. Convertendo review_date para formato de data padrão (DateType)
reviews_df_silver = reviews_df_silver.withColumn("reviewTime", to_date("reviewTime", "MM-dd-yyyy"))

# 5. Filtrando as reviews de 2013 a 2018
reviews_df_silver = reviews_df_silver.filter("reviewTime >= '2013-01-01' and reviewTime <= '2018-12-31'")

# 6. Selecionar apenas as colunas que são importantes para a análise
columns_to_keep = ["overall", "reviewTime", "reviewerID", "asin", "reviewerName", "reviewText"]
reviews_df_silver = reviews_df_silver.select(*columns_to_keep)

# 7. Remover linhas com valores nulos nas colunas importantes
reviews_df_silver = reviews_df_silver.na.drop(subset=["overall", "reviewText"])

# 8. Removendo caracteres especiais de cada valor da coluna 'reviewText'.
reviews_df_silver = reviews_df_silver.withColumn("reviewText", regexp_replace("reviewText", "[^a-zA-Z0-9\\s]", ""))

# 9. Remove os espaços em branco que estiverem no início e no fim de cada valor da coluna 'reviewText'.
reviews_df_silver = reviews_df_silver.withColumn("reviewText", trim(col("reviewText")))

# 10. Converte todo o texto da coluna 'reviewText' para letras minúsculas.
reviews_df_silver = reviews_df_silver.withColumn("reviewText", lower(col("reviewText")))

#### -------------------------------------- Enriquecimento ---------------------------------------- ####

# 1. Extração de ano e mês da data de 'review'
reviews_df_silver = reviews_df_silver.withColumn("year", year(col("reviewTime")))
reviews_df_silver = reviews_df_silver.withColumn("month", month(col("reviewTime")))

# 2. Classificação da Avaliação
# 2.1. Criar uma coluna de sentimento baseada em 'overall'
reviews_df_silver = reviews_df_silver.withColumn("sentiment",
                                                  when(col("overall") >= 4, "positive"). # Positivo
                                                  when(col("overall") == 3, "neutral"). # Neutro
                                                  otherwise("negative")) # Negativo

# 2.2. Criar coluna de sentimento com valores inteiros para análise (2 para positivo, 1 para neutro e 0 para negativo)
reviews_df_silver = reviews_df_silver.withColumn("sentimentOverall",
                                                  when(col("overall") >= 4, 2). # Positivo
                                                  when(col("overall") == 3, 1). # Neutro
                                                  otherwise(0) # Negativo
)

print()
print("Dados enriquecidos:")
print()

# Exibir schema dos dados carregados
reviews_df_silver.printSchema()

# Exibir as primeiras 10 linhas dos dados transformados de 'reviews'
reviews_df_silver.show(10)

#### ------------------------------------------------------------------------------------------- ####

# Selecionar apenas as colunas que são importantes para a análise
columns_to_keep = ["reviewText", "sentimentOverall"]
reviews_df_silver = reviews_df_silver.select(*columns_to_keep)

# Exibir schema dos dados carregados
print()
print("Dados finalizados:")
print()
reviews_df_silver.printSchema()

# Exibir as primeiras 10 linhas dos dados transformados de 'reviews'
reviews_df_silver.show(10)

# Salva o DataFrame como tabela da camada Silver (formato .parquet)
# obs: Altere o caminho do arquivo para sua preferência, desde que esteja em seu Drive ou GCS.
reviews_df_silver.coalesce(1).write.mode("overwrite").option("header", True).parquet("/content/drive/MyDrive/Disciplinas-UFG/PDM/TrabalhoFinal/ArquiteturaMedallion/Silver/reviews_silver")


Dados enriquecidos:

root
 |-- overall: float (nullable = true)
 |-- reviewTime: date (nullable = true)
 |-- reviewerID: string (nullable = true)
 |-- asin: string (nullable = true)
 |-- reviewerName: string (nullable = true)
 |-- reviewText: string (nullable = false)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- sentiment: string (nullable = false)
 |-- sentimentOverall: integer (nullable = false)

+-------+----------+--------------+----------+-----------------+--------------------+----+-----+---------+----------------+
|overall|reviewTime|    reviewerID|      asin|     reviewerName|          reviewText|year|month|sentiment|sentimentOverall|
+-------+----------+--------------+----------+-----------------+--------------------+----+-----+---------+----------------+
|    2.0|2014-04-14|A3J3BRHTDRFJ2G|0511189877|         EJ Honda|this remote for w...|2014|    4| negative|               0|
|    5.0|2017-05-31| A7362LXMQEM6W|0511189877|   Online shopper|wo